# **Child Well-being - Data Extractor**<br/>
**University: University of Milano-Bicocca**<br/>
**Master's Degree: Data Science (A.Y. 2025/2026)**<br/>
**Course: Data Science Lab**<br/>

In [1]:
from io import StringIO

import requests
import polars as pl

In [2]:
# Define API query URL (CSV with labels format)
url = 'https://sdmx.oecd.org/public/rest/data/OECD.WISE.CWB,DSD_CWB@DF_CWB,/all?dimensionAtObservation=AllDimensions&format=csvfile'
# Fetch data
response = requests.get(url)

# Load into polars DataFrame
df = pl.read_csv(StringIO(response.text))

In [3]:
# Filters from a preliminary analysis of missing values and data availability across dimensions, measures, countries, and years

## Keep only the measures with the lowest number of missing values for the years 2009,2012,2015,2018
measures = [
    'A1_2','A1_4',                              # Material conditions
    'A2_1',                                     # Physical health conditions
    'A3_3','A3_4','A3_5',                       # Cognitive & educational outcome
    'A4_6',                                     # Social & emotional outcome
    'B1_1','B1_5',                              # Home & Family Life
    'B2_1','B2_4','B2_5',                       # Life at school & early education and care
    'B3_5',                                     # Social Life and life in the community
    'B4_3',                                     # Online life
    'C1_1','C1_2','C1_3','C1_4','C1_5',         # Family policies
    'C2_1','C2_2','C2_3',                       # House & Community policies
    'C3_1','C3_2','C3_3',                       # Health policies
    'C4_1','C4_2','C4_3','C4_4','C4_5','C4_6',  # Education & early childhood policies
    'C5_1',                                     # Environmental policies
]

## Keep only the European countries
areas = [
    'SWE', 'DNK', 'NOR', 'FIN', 'ISL', 'LTU', 'LVA', 'EST',        # Northern Europe
    'IRL', 'GBR', 'NLD', 'BEL', 'FRA', 'LUX', 'CHE', 'AUT', 'DEU', # Western Europe
    'ESP', 'PRT', 'ITA', 'GRC', 'SVN', 'HRV', 'TUR',               # Southern Europe
    'POL', 'CZE', 'SVK', 'HUN', 'ROU', 'BGR',                      # Eastern Europe
]

## Keep only a subset of years
years = [2009, 2012, 2015, 2018]

In [4]:
# Apply the filters on the dataframe
df = df.filter(
    pl.col('MEASURE').is_in(measures) &
    pl.col('REF_AREA').is_in(areas) &
    pl.col('TIME_PERIOD').is_in(years)
)

In [5]:
df.head(10)

DATAFLOW,REF_AREA,MEASURE,DOMAIN,TIME_PERIOD,OBS_VALUE,OBS_STATUS,UNIT_MULT,UNIT_MEASURE,BASE_PER,DECIMALS,POP_GROUP
str,str,str,str,i64,f64,str,i64,str,i64,i64,str
"""OECD.WISE.CWB:DSD_CWB@DF_CWB(1…","""AUT""","""C2_1""","""C2""",2009,204.6,"""A""",0,"""USD_PPP""",2015,1,"""_Z"""
"""OECD.WISE.CWB:DSD_CWB@DF_CWB(1…","""AUT""","""C2_1""","""C2""",2012,186.1,"""A""",0,"""USD_PPP""",2015,1,"""_Z"""
"""OECD.WISE.CWB:DSD_CWB@DF_CWB(1…","""AUT""","""C2_1""","""C2""",2015,175.4,"""A""",0,"""USD_PPP""",2015,1,"""_Z"""
"""OECD.WISE.CWB:DSD_CWB@DF_CWB(1…","""AUT""","""C2_1""","""C2""",2018,169.8,"""A""",0,"""USD_PPP""",2015,1,"""_Z"""
"""OECD.WISE.CWB:DSD_CWB@DF_CWB(1…","""BEL""","""C2_1""","""C2""",2009,182.4,"""A""",0,"""USD_PPP""",2015,1,"""_Z"""
"""OECD.WISE.CWB:DSD_CWB@DF_CWB(1…","""BEL""","""C2_1""","""C2""",2012,184.7,"""A""",0,"""USD_PPP""",2015,1,"""_Z"""
"""OECD.WISE.CWB:DSD_CWB@DF_CWB(1…","""BEL""","""C2_1""","""C2""",2015,130.7,"""A""",0,"""USD_PPP""",2015,1,"""_Z"""
"""OECD.WISE.CWB:DSD_CWB@DF_CWB(1…","""BEL""","""C2_1""","""C2""",2018,162.4,"""A""",0,"""USD_PPP""",2015,1,"""_Z"""
"""OECD.WISE.CWB:DSD_CWB@DF_CWB(1…","""CZE""","""C2_1""","""C2""",2009,305.8,"""A""",0,"""USD_PPP""",2015,1,"""_Z"""


In [6]:
# Create a grid with all the combinations of area and time period
df_grid = pl.DataFrame()

years = df.select('TIME_PERIOD').unique().to_series().to_list()

for area in areas:
    for year in years:
        df_grid = df_grid.vstack(
            pl.DataFrame(
                {
                    'REF_AREA': area,
                    'TIME_PERIOD': year,
                }
            )
        )

#Populate the grid with the measures in a wide format
for measure in measures:
    df_temp = df.filter(
        pl.col('MEASURE') == measure
    ).select(
        ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
    ).with_columns(
        pl.col('OBS_VALUE').alias(measure)
    ).select(
        ['REF_AREA', 'TIME_PERIOD', measure]
    )
    df_grid = df_grid.join(
        df_temp,
        on=['REF_AREA', 'TIME_PERIOD'],
        how='left'
    )

In [7]:
df_grid.head(10)

REF_AREA,TIME_PERIOD,A1_2,A1_4,A2_1,A3_3,A3_4,A3_5,A4_6,B1_1,B1_5,B2_1,B2_4,B2_5,B3_5,B4_3,C1_1,C1_2,C1_3,C1_4,C1_5,C2_1,C2_2,C2_3,C3_1,C3_2,C3_3,C4_1,C4_2,C4_3,C4_4,C4_5,C4_6,C5_1
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""SWE""",2015,4.4,1.7,2.5,16.7,58.9,9.1,null,9.1,59.5,null,17.9,69.3,12.4,37.4,9818.3,7.0,37.0,60.0,10.0,289.1,621.9,155.9,4201.5,98.0,98.0,10716.7,5.0,6.4,11581.8,1220.3,13.4,208.7
"""SWE""",2018,5.0,1.0,2.0,19.4,69.8,8.9,32.6,9.0,46.4,46.3,19.2,67.0,15.5,33.5,null,6.9,37.0,55.7,14.3,362.5,635.5,143.3,4389.6,97.0,97.0,null,5.0,6.3,12368.5,1254.5,12.3,252.8
"""SWE""",2012,2.4,0.7,2.6,12.4,null,9.7,null,null,null,null,null,null,11.7,null,9748.8,null,39.0,60.0,10.0,242.6,603.6,154.0,3960.1,98.0,98.0,10194.2,6.0,null,11262.5,null,null,214.1
"""SWE""",2009,null,1.5,2.5,15.9,null,11.0,null,null,null,null,null,null,null,null,9185.3,null,41.0,60.0,10.0,259.4,580.5,134.9,3026.3,98.0,97.0,9525.1,null,null,10602.2,null,null,204.8
"""DNK""",2015,4.1,0.5,3.7,14.9,40.6,11.4,null,2.9,50.4,null,20.1,70.3,5.3,46.5,10226.9,10.1,63.0,50.0,2.0,115.7,863.5,345.1,3934.1,90.0,88.0,10179.8,11.0,null,null,null,null,207.5
"""DNK""",2018,4.1,0.3,3.7,15.8,64.0,12.0,null,4.9,46.4,56.0,21.4,72.0,5.2,42.7,null,7.2,58.0,50.0,2.0,119.6,845.9,342.7,4084.6,96.0,94.0,null,11.0,6.8,10140.2,43.8,11.3,206.6
"""DNK""",2012,2.8,0.4,3.4,12.5,null,13.3,null,2.7,null,null,null,null,8.6,null,10300.6,9.6,63.0,50.0,2.0,133.8,871.2,331.5,3755.0,91.0,87.0,10408.6,12.0,null,11820.4,null,null,189.9
"""DNK""",2009,null,1.1,3.1,13.9,null,8.8,null,null,null,null,null,null,null,null,10374.4,null,61.0,50.0,2.0,234.4,866.0,313.9,3824.1,89.0,84.0,9424.0,null,null,12717.1,null,null,204.8
"""NOR""",2015,0.8,1.1,2.2,17.6,61.4,9.2,null,7.3,57.0,null,17.6,75.7,4.8,null,11380.7,6.0,39.0,91.0,10.0,456.5,983.4,74.2,4897.3,95.0,95.0,11152.0,7.0,16.0,13993.4,492.8,9.9,534.8


In [8]:
df_grid.write_parquet('../data/010_child_well_being.parquet')

*This notebook is licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*